20260215 주식실적발표 관련된 내용들을 취합할 수 있게 만들었음

이거 나중에 종합해서 붙일 수 있도록 해야함

네, 기존 05_check_earnings.ipynb 파일은 "이미 매수 추천된 종목"만 확인하는 구조였는데, 이를 **"모든 종목의 실적 일정을 미리 수집하여 데이터베이스(CSV)로 만드는 역할"**로 변경해야 합니다.

그래야 분석 코드(02_1_...)가 실행될 때 이 파일을 참고해서 모든 종목의 실적 D-Day를 계산할 수 있습니다.

In [2]:
import pandas as pd
import yfinance as yf
import os
import glob
from tqdm import tqdm
from datetime import datetime

# ==========================================
# ⚙️ 설정
# ==========================================
MARKET = 'US_ALL'
RAW_DATA_DIR = f"./Raw_Data/{MARKET}"
OUTPUT_FILE = "earnings_calendar.csv" # 최종 저장할 파일명

# ==========================================
# 🛠️ 함수 정의
# ==========================================

def get_next_earnings_date(ticker):
    """
    yfinance를 이용해 다음 실적 발표일을 가져옴 (YYYY-MM-DD 문자열 반환)
    """
    try:
        stock = yf.Ticker(ticker)
        
        # 1. Calendar 속성 확인 (가장 정확한 예정일)
        cal = stock.calendar
        
        earnings_date = None
        
        # yfinance 버전에 따라 반환 타입(Dict 또는 DataFrame)이 다를 수 있어 처리
        if isinstance(cal, dict) and 'Earnings Date' in cal:
            dates = cal['Earnings Date']
            if len(dates) > 0:
                earnings_date = dates[0]
        elif isinstance(cal, pd.DataFrame):
            if not cal.empty:
                # Transpose 되어 있는 경우 등 다양한 구조 대응
                if 'Earnings Date' in cal.index:
                    earnings_date = cal.loc['Earnings Date'].iloc[0]
                elif 0 in cal.columns:
                     earnings_date = cal.iloc[0, 0]

        # 2. 날짜 포맷 변환
        if earnings_date:
            # datetime 객체라면 변환
            if pd.api.types.is_datetime64_any_dtype(earnings_date):
                return earnings_date.strftime('%Y-%m-%d')
            # 문자열이거나 다른 객체라면 변환 시도
            return datetime.strptime(str(earnings_date), "%Y-%m-%d").strftime('%Y-%m-%d')

    except Exception:
        pass
    
    return "-" # 정보 없음

def run_create_calendar():
    # 1. 분석 대상 종목 파일 수집
    if not os.path.exists(RAW_DATA_DIR):
        print(f"❌ 데이터 폴더가 없습니다: {RAW_DATA_DIR}")
        return

    files = glob.glob(f"{RAW_DATA_DIR}/*.csv")
    files = [f for f in files if "sector_map" not in f] # 섹터 맵 파일 제외
    
    print(f"🚀 [Earnings Calendar] 전체 {len(files)}개 종목 실적 발표일 DB 생성 시작...")
    print("   (네트워크 통신이 필요하여 시간이 다소 걸립니다.)")
    
    results = []
    
    # 2. 전체 종목 순회하며 실적일 수집
    for file_path in tqdm(files):
        try:
            filename = os.path.basename(file_path).replace('.csv', '')
            parts = filename.split('_')
            code = parts[0] # 종목 코드 추출 (예: AAPL)
            
            # 실적 발표일 조회 (yfinance)
            next_date = get_next_earnings_date(code)
            
            results.append({
                'Ticker': code,
                'Next_Earnings': next_date
            })
        except Exception:
            continue
            
    # 3. 결과 저장
    if results:
        df = pd.DataFrame(results)
        df.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')
        print(f"\n✅ 실적 달력 생성 완료: {OUTPUT_FILE}")
        print(f"   - 총 {len(df)}개 종목 정보 저장됨")
        print("   - 이제 02_1_nasdaq_analysis.ipynb를 실행하면 이 파일을 자동으로 읽어옵니다.")
    else:
        print("\n❌ 저장할 데이터가 없습니다.")

if __name__ == "__main__":
    run_create_calendar()

🚀 [Earnings Calendar] 전체 2221개 종목 실적 발표일 DB 생성 시작...
   (네트워크 통신이 필요하여 시간이 다소 걸립니다.)


100%|██████████| 2221/2221 [08:44<00:00,  4.23it/s]


✅ 실적 달력 생성 완료: earnings_calendar.csv
   - 총 2221개 종목 정보 저장됨
   - 이제 02_1_nasdaq_analysis.ipynb를 실행하면 이 파일을 자동으로 읽어옵니다.
